# Malawi Landslide Mapping — Dual-Branch SegFormer MiT-B2 Training

Reproducible training workflow for RGB imagery with pre-event and post-event VV coherence.
**Required dataset folders:** `RGB`, `VV_COH_pre`, `VV_COH_post`, and `Mask` under the `train` and `val` splits.


In [ ]:
# =========================================================
# CELL 1 — Imports and optional Hugging Face login
# =========================================================

# If needed:
# !pip install -q transformers datasets evaluate accelerate opencv-python pillow safetensors

from huggingface_hub import notebook_login
# notebook_login()

import os, re, csv, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image as PILImage
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset
import evaluate

from transformers import (
    AutoImageProcessor,
    AutoModelForSemanticSegmentation,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from transformers.modeling_outputs import SemanticSegmenterOutput

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# =========================================================
# CELL 2 — Reproducibility
# =========================================================

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

seed_everything(SEED)


In [ ]:
# =========================================================
# CELL 3 — Configuration
# =========================================================

checkpoint = "nvidia/mit-b2"

# Set these paths before execution.
DATASET_ROOT = Path(r"")
OUTPUT_DIR = Path(r"")

if str(DATASET_ROOT) in ("", "."):
    raise ValueError("Set DATASET_ROOT to the dataset directory containing train/ and val/.")
if str(OUTPUT_DIR) in ("", "."):
    raise ValueError("Set OUTPUT_DIR to the directory where training outputs will be written.")

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"

train_rgb_dir      = TRAIN_DIR / "RGB"
train_coh_pre_dir  = TRAIN_DIR / "VV_COH_pre"
train_coh_post_dir = TRAIN_DIR / "VV_COH_post"
train_mask_dir     = TRAIN_DIR / "Mask"

val_rgb_dir      = VAL_DIR / "RGB"
val_coh_pre_dir  = VAL_DIR / "VV_COH_pre"
val_coh_post_dir = VAL_DIR / "VV_COH_post"
val_mask_dir     = VAL_DIR / "Mask"

output_dir = str(OUTPUT_DIR)
metrics_file = "training_metrics.csv"
metrics_file_path = os.path.join(output_dir, metrics_file)
best_model_dir = os.path.join(output_dir, "best_model")
os.makedirs(output_dir, exist_ok=True)

id2label = {0: "background", 1: "landslide"}
label2id = {"background": 0, "landslide": 1}
num_labels = 2

CE_WEIGHT = 1.0
TVERSKY_WEIGHT = 1.0
TVERSKY_ALPHA = 0.6
TVERSKY_BETA = 0.4

LAMBDA_COH_BCE = 0.05
LAMBDA_COH_CONSISTENCY = 0.05

AUX_IN_CHANNELS = 2
AUX_NAMES = ["COH_pre", "COH_post"]

# Four orientations used by the original augmented training set.
TRAIN_ROTATIONS = (0, 90, 180, 270)

print("Checkpoint:", checkpoint)
print("Dataset root:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)
print("Auxiliary inputs:", AUX_NAMES)
print("Training rotations:", TRAIN_ROTATIONS)
print("Tversky alpha/beta:", TVERSKY_ALPHA, TVERSKY_BETA)
print("Coherence prior guidance weights:", LAMBDA_COH_BCE, LAMBDA_COH_CONSISTENCY)

In [ ]:
# =========================================================
# CELL 4 — File matching helpers
# =========================================================

IMG_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")

def get_tile_id(filename):
    base = os.path.splitext(os.path.basename(str(filename)))[0]

    prefixes = [
        "RGB_", "rgb_",
        "COH_pre_", "coh_pre_",
        "COH_post_", "coh_post_",
        "Mask_", "mask_",
        "Label_", "label_",
        "Annotation_", "annotation_",
        "Image_", "image_",
    ]
    for p in prefixes:
        if base.startswith(p):
            base = base.replace(p, "", 1)
            break

    for pattern in [r"(neg_tile_\d+_rot\d+)", r"(neg_tile_\d+)", r"(tile_\d+_rot\d+)", r"(tile_\d+)"]:
        m = re.search(pattern, base, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return base

def list_image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS])

def build_index(folder):
    return {get_tile_id(p.name): p for p in list_image_files(folder)}

def resolve_mask_dir(primary_dir, fallback_dirs, split_name):
    candidates = [Path(primary_dir)] + [Path(p) for p in fallback_dirs]
    print(f"\nMask search for {split_name}:")
    for c in candidates:
        n = len(list_image_files(c))
        print(f"  {c} -> {n} files")
        if c.exists() and n > 0:
            print(f"  Using mask folder: {c}")
            return c
    print(f"  No valid mask folder found for {split_name}.")
    return Path(primary_dir)

In [ ]:
# =========================================================
# CELL 5 — Build training and validation datasets
# =========================================================

def read_rgb(path):
    return PILImage.open(path).convert("RGB")

def read_mask(path, image_path=None, force_black=False):
    if force_black:
        if image_path is not None:
            img = PILImage.open(image_path).convert("RGB")
            w, h = img.size
            return PILImage.fromarray(np.zeros((h, w), dtype=np.uint8), mode="L")
        return PILImage.fromarray(np.zeros((128, 128), dtype=np.uint8), mode="L")

    mask = PILImage.open(path).convert("L")
    mask_np = (np.array(mask) > 0).astype(np.uint8)
    return PILImage.fromarray(mask_np, mode="L")

def collect_samples(rgb_dir, coh_pre_dir, coh_post_dir, mask_dir, split_name):
    rgb_idx = build_index(rgb_dir)
    coh_pre_idx = build_index(coh_pre_dir)
    coh_post_idx = build_index(coh_post_dir)
    mask_idx = build_index(mask_dir)

    ids = sorted(set(rgb_idx) & set(coh_pre_idx) & set(coh_post_idx) & set(mask_idx))

    print("\n" + "=" * 80)
    print(split_name)
    print("=" * 80)
    print("RGB     :", len(rgb_idx), rgb_dir)
    print("COH_pre :", len(coh_pre_idx), coh_pre_dir)
    print("COH_post:", len(coh_post_idx), coh_post_dir)
    print("Mask    :", len(mask_idx), mask_dir)
    print("Matched :", len(ids))

    if len(ids) == 0:
        raise ValueError(f"No matched samples found for {split_name}")

    data = []
    for tid in tqdm(ids, desc=f"Loading {split_name}"):
        rgb_path = rgb_idx[tid]
        data.append({
            "id": tid,
            "image": read_rgb(rgb_path),
            "annotation": read_mask(mask_idx[tid], image_path=rgb_path),
            "coh_pre_path": str(coh_pre_idx[tid]),
            "coh_post_path": str(coh_post_idx[tid]),
            "is_negative": 0,
        })
    return data

train_original = collect_samples(
    train_rgb_dir, train_coh_pre_dir, train_coh_post_dir,
    train_mask_dir, "Train"
)

valid_data = collect_samples(
    val_rgb_dir, val_coh_pre_dir, val_coh_post_dir,
    val_mask_dir, "Validation"
)

# Reconstruct the original four-orientation training set in memory.
# Virtual augmented IDs are sorted lexicographically so that dataset indexing
# follows the ordering produced when the augmented files are physically stored
# and collected with sorted tile IDs.
train_data = []
for sample in train_original:
    for angle in TRAIN_ROTATIONS:
        item = dict(sample)
        item["rotation"] = int(angle)
        item["id"] = f'{sample["id"]}_rot{angle}'
        train_data.append(item)

train_data = sorted(train_data, key=lambda x: x["id"])

# Validation is never augmented.
for sample in valid_data:
    sample["rotation"] = 0

train_ds = Dataset.from_list(train_data)
valid_ds = Dataset.from_list(valid_data)

print("\nDataset summary")
print("Original training samples :", len(train_original))
print("Training samples after rotation reconstruction:", len(train_ds))
print("Validation samples        :", len(valid_ds))

expected_train = len(train_original) * len(TRAIN_ROTATIONS)
assert len(train_ds) == expected_train
assert all(int(x["rotation"]) == 0 for x in valid_data)

In [ ]:
# =========================================================
# CELL 6 — Image processor and RGB+COH transforms
# =========================================================

image_processor = AutoImageProcessor.from_pretrained(checkpoint, use_fast=False)

def read_single_band_layer(path, target_hw):
    arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if arr is None:
        arr = np.array(PILImage.open(path).convert("L"))
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    arr = arr.astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if arr.max() <= 1.5 and arr.min() >= -0.5:
        arr = np.clip(arr, 0, 1)
    else:
        p1, p99 = np.percentile(arr, [1, 99])
        if p99 > p1:
            arr = (arr - p1) / (p99 - p1)
        else:
            arr = np.zeros_like(arr, dtype=np.float32)
        arr = np.clip(arr, 0, 1)
    h, w = target_hw
    if arr.shape[:2] != (h, w):
        arr = cv2.resize(arr, (w, h), interpolation=cv2.INTER_LINEAR)
    return arr.astype(np.float32)

def rotate_array_90(arr, angle):
    """Rotate by 0/90/180/270 degrees without interpolation."""
    k = (int(angle) // 90) % 4
    return np.rot90(arr, k=k).copy() if k else arr

def apply_transforms(example_batch):
    images = []
    labels = []
    aux_values = []

    for image, annotation, coh_pre_path, coh_post_path, rotation in zip(
        example_batch["image"],
        example_batch["annotation"],
        example_batch["coh_pre_path"],
        example_batch["coh_post_path"],
        example_batch["rotation"],
    ):
        angle = int(rotation)

        image_np = np.array(image)
        label_np = np.array(annotation)

        image_np = rotate_array_90(image_np, angle)
        label_np = rotate_array_90(label_np, angle)

        image_rot = PILImage.fromarray(image_np).convert("RGB")
        label_rot = PILImage.fromarray(label_np.astype(np.uint8)).convert("L")

        processed = image_processor(
            images=[image_rot],
            segmentation_maps=[label_rot]
        )

        pixel_values = processed["pixel_values"][0]
        processed_label = processed["labels"][0]
        h, w = pixel_values.shape[-2], pixel_values.shape[-1]

        coh_pre = read_single_band_layer(coh_pre_path, image_np.shape[:2])
        coh_post = read_single_band_layer(coh_post_path, image_np.shape[:2])
        coh_pre = rotate_array_90(coh_pre, angle)
        coh_post = rotate_array_90(coh_post, angle)

        if coh_pre.shape != (h, w):
            coh_pre = cv2.resize(coh_pre, (w, h), interpolation=cv2.INTER_LINEAR)
        if coh_post.shape != (h, w):
            coh_post = cv2.resize(coh_post, (w, h), interpolation=cv2.INTER_LINEAR)

        images.append(pixel_values)
        labels.append(processed_label)
        aux_values.append(
            np.stack([coh_pre, coh_post], axis=0).astype(np.float32)
        )

    return {
        "pixel_values": images,
        "labels": labels,
        "aux_values": aux_values,
    }

train_ds.set_transform(apply_transforms)
valid_ds.set_transform(apply_transforms)

In [ ]:
# =========================================================
# CELL 7 — Visual sanity check
# =========================================================

example = train_ds[0]
pixel_values = np.array(example["pixel_values"])
aux_values = np.array(example["aux_values"])
labels = np.array(example["labels"])

rgb_vis = np.transpose(pixel_values, (1, 2, 0))
rgb_vis = (rgb_vis - rgb_vis.min()) / (rgb_vis.max() - rgb_vis.min() + 1e-6)
coh_diff_abs = np.abs(aux_values[1] - aux_values[0])

fig, axes = plt.subplots(1, 6, figsize=(24, 4))
axes[0].imshow(rgb_vis); axes[0].set_title("RGB after processor"); axes[0].axis("off")
axes[1].imshow(aux_values[0], cmap="gray", vmin=0, vmax=1); axes[1].set_title("COH_pre"); axes[1].axis("off")
axes[2].imshow(aux_values[1], cmap="gray", vmin=0, vmax=1); axes[2].set_title("COH_post"); axes[2].axis("off")
axes[3].imshow(coh_diff_abs, cmap="magma", vmin=0, vmax=1); axes[3].set_title("|COH_post - COH_pre|"); axes[3].axis("off")
axes[4].imshow(labels, cmap="gray", vmin=0, vmax=1); axes[4].set_title("Mask"); axes[4].axis("off")
axes[5].imshow(rgb_vis); axes[5].imshow(labels, cmap="Reds", alpha=0.35); axes[5].set_title("Overlay"); axes[5].axis("off")
plt.tight_layout(); plt.show()

print("ID:", example["id"])
print("Rotation:", example["rotation"])
print("RGB pixel_values shape:", pixel_values.shape)
print("Auxiliary shape:", aux_values.shape, "-> [COH_pre, COH_post]")
print("Labels shape:", labels.shape)
print("Unique label values:", np.unique(labels))


In [ ]:
# =========================================================
# CELL 8 — Residual dual-branch SegFormer with coherence-change prior and Tversky loss
# =========================================================

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch)) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        identity = self.skip(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class ResidualCoherenceEncoder(nn.Module):
    def __init__(self, in_channels=2, channels=(32, 64, 128, 256)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = nn.Sequential(nn.Conv2d(in_channels, c1, 3, stride=1, padding=1, bias=False), nn.BatchNorm2d(c1), nn.ReLU(inplace=True))
        self.rb0 = ResidualBlock(c1, c1, stride=2)  # H/2
        self.rb1 = ResidualBlock(c1, c1, stride=2)  # H/4
        self.rb2 = ResidualBlock(c1, c2, stride=2)  # H/8
        self.rb3 = ResidualBlock(c2, c3, stride=2)  # H/16
        self.rb4 = ResidualBlock(c3, c4, stride=2)  # H/32
    def forward(self, x):
        x = self.stem(x)
        x = self.rb0(x)
        c1 = self.rb1(x)
        c2 = self.rb2(c1)
        c3 = self.rb3(c2)
        c4 = self.rb4(c3)
        return [c1, c2, c3, c4]

class CoherenceGuidedStageFusion(nn.Module):
    def __init__(self, rgb_channels, aux_channels):
        super().__init__()
        self.aux_proj = nn.ModuleList([nn.Sequential(nn.Conv2d(a, r, 1, bias=False), nn.BatchNorm2d(r), nn.ReLU(inplace=True)) for r, a in zip(rgb_channels, aux_channels)])
        self.gates = nn.ModuleList([nn.Sequential(nn.Conv2d(r * 2, r, 1, bias=True), nn.Sigmoid()) for r in rgb_channels])
    def forward(self, rgb_feats, aux_feats):
        fused = []
        for rgb, aux, proj, gate_layer in zip(rgb_feats, aux_feats, self.aux_proj, self.gates):
            aux = proj(aux)
            if aux.shape[-2:] != rgb.shape[-2:]:
                aux = F.interpolate(aux, size=rgb.shape[-2:], mode="bilinear", align_corners=False)
            gate = gate_layer(torch.cat([rgb, aux], dim=1))
            fused.append(rgb + gate * aux)
        return tuple(fused)

class CoherencePriorHead(nn.Module):
    def __init__(self, aux_channels=(32, 64, 128, 256), mid_ch=64):
        super().__init__()
        self.proj = nn.ModuleList([nn.Sequential(nn.Conv2d(ch, mid_ch, 1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True)) for ch in aux_channels])
        self.head = nn.Sequential(nn.Conv2d(mid_ch * 4, mid_ch, 3, padding=1, bias=False), nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True), nn.Conv2d(mid_ch, 1, 1))
    def forward(self, aux_feats, output_size):
        ups = []
        for feat, proj in zip(aux_feats, self.proj):
            p = proj(feat)
            p = F.interpolate(p, size=output_size, mode="bilinear", align_corners=False)
            ups.append(p)
        return self.head(torch.cat(ups, dim=1))

class SegFormerRGBCoherenceDualBranch(nn.Module):
    def __init__(self, checkpoint, num_labels, id2label, label2id, aux_in_channels=2,
                 ce_weight=1.0, tversky_weight=1.0, tversky_alpha=0.6, tversky_beta=0.4,
                 lambda_coh_bce=0.05, lambda_coh_consistency=0.05):
        super().__init__()
        self.segformer = AutoModelForSemanticSegmentation.from_pretrained(
            checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True, use_safetensors=True
        )
        self.aux_channels = (32, 64, 128, 256)
        self.aux_encoder = ResidualCoherenceEncoder(in_channels=aux_in_channels, channels=self.aux_channels)
        self.fusion = CoherenceGuidedStageFusion(list(self.segformer.config.hidden_sizes), self.aux_channels)
        self.coh_prior_head = CoherencePriorHead(aux_channels=self.aux_channels, mid_ch=64)
        self.config = self.segformer.config
        self.ce_weight = ce_weight
        self.tversky_weight = tversky_weight
        self.tversky_alpha = tversky_alpha
        self.tversky_beta = tversky_beta
        self.lambda_coh_bce = lambda_coh_bce
        self.lambda_coh_consistency = lambda_coh_consistency
    @property
    def device(self):
        return next(self.parameters()).device
    def tversky_loss_binary(self, logits, labels, eps=1e-7):
        probs = torch.softmax(logits, dim=1)[:, 1]
        target = (labels == 1).float()
        tp = (probs * target).sum(dim=(1, 2))
        fp = (probs * (1.0 - target)).sum(dim=(1, 2))
        fn = ((1.0 - probs) * target).sum(dim=(1, 2))
        tversky = (tp + eps) / (tp + self.tversky_alpha * fp + self.tversky_beta * fn + eps)
        return 1.0 - tversky.mean()
    def forward(self, pixel_values=None, aux_values=None, labels=None, **kwargs):
        if aux_values is None:
            raise ValueError("aux_values is required: [B,2,H,W] = [COH_pre, COH_post].")
        enc = self.segformer.segformer(pixel_values, output_hidden_states=True, return_dict=True)
        rgb_hidden_states = enc.hidden_states
        aux_feats = self.aux_encoder(aux_values)
        fused_hidden_states = self.fusion(rgb_hidden_states, aux_feats)
        logits = self.segformer.decode_head(fused_hidden_states)
        loss = None
        if labels is not None:
            up_logits = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
            ce_loss = F.cross_entropy(up_logits, labels.long(), ignore_index=255)
            tv_loss = self.tversky_loss_binary(up_logits, labels.long())
            main_loss = self.ce_weight * ce_loss + self.tversky_weight * tv_loss
            coh_logits = self.coh_prior_head(aux_feats, output_size=labels.shape[-2:])
            coh_bce_loss = F.binary_cross_entropy_with_logits(coh_logits[:, 0], (labels == 1).float())
            seg_prob = torch.softmax(up_logits, dim=1)[:, 1]
            coh_prob = torch.sigmoid(coh_logits[:, 0]).detach()
            coh_consistency_loss = F.mse_loss(seg_prob, coh_prob)
            loss = main_loss + self.lambda_coh_bce * coh_bce_loss + self.lambda_coh_consistency * coh_consistency_loss
        return SemanticSegmenterOutput(loss=loss, logits=logits, hidden_states=fused_hidden_states, attentions=None)

model = SegFormerRGBCoherenceDualBranch(
    checkpoint=checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    aux_in_channels=AUX_IN_CHANNELS,
    ce_weight=CE_WEIGHT,
    tversky_weight=TVERSKY_WEIGHT,
    tversky_alpha=TVERSKY_ALPHA,
    tversky_beta=TVERSKY_BETA,
    lambda_coh_bce=LAMBDA_COH_BCE,
    lambda_coh_consistency=LAMBDA_COH_CONSISTENCY,
).to(device)

print("RGB + COH_pre + COH_post residual dual-branch SegFormer created.")
print("MiT hidden sizes:", model.segformer.config.hidden_sizes)
print("Auxiliary inputs:", AUX_NAMES)
print("Loss: CE + Tversky(alpha=0.6,beta=0.4) + coherence prior BCE + consistency")
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# =========================================================
# CELL 9 — Metrics
# =========================================================

metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    logits_tensor = torch.from_numpy(logits)
    labels_tensor = torch.from_numpy(labels)
    logits_tensor = F.interpolate(logits_tensor, size=labels_tensor.shape[-2:], mode="bilinear", align_corners=False)
    preds = logits_tensor.argmax(dim=1).cpu().numpy()
    labels_np = labels_tensor.cpu().numpy()
    pred_fg = preds == 1
    label_fg = labels_np == 1
    tp = np.logical_and(pred_fg, label_fg).sum()
    fp = np.logical_and(pred_fg, ~label_fg).sum()
    fn = np.logical_and(~pred_fg, label_fg).sum()
    tn = np.logical_and(~pred_fg, ~label_fg).sum()
    eps = 1e-7
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)
    background_iou = tn / (tn + fp + fn + eps)
    miou = (background_iou + iou) / 2
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    return {"F1": float(f1), "Recall": float(recall), "Precision": float(precision), "IoU": float(iou), "mIoU": float(miou), "Accuracy": float(accuracy)}


In [ ]:
# =========================================================
# CELL 10 — Display and metrics callbacks
# =========================================================

class DisplayCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, eval_dataloader=None, **kwargs):
        if eval_dataloader is None or state.global_step % 1000 != 0:
            return
        batch = next(iter(eval_dataloader))
        inputs = batch["pixel_values"].to(model.device)
        aux_values = batch["aux_values"].to(model.device)
        labels = batch["labels"].to(model.device)
        with torch.no_grad():
            outputs = model(pixel_values=inputs, aux_values=aux_values)
            logits = F.interpolate(outputs.logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
            preds = logits.argmax(dim=1)
            aux_feats = model.aux_encoder(aux_values)
            coh_logits = model.coh_prior_head(aux_feats, output_size=labels.shape[-2:])
            coh_prob = torch.sigmoid(coh_logits[:, 0])
        image = inputs[0].detach().cpu().permute(1, 2, 0).numpy()
        image = (image - image.min()) / (image.max() - image.min() + 1e-6)
        coh_pre = aux_values[0, 0].detach().cpu().numpy()
        coh_post = aux_values[0, 1].detach().cpu().numpy()
        coh_diff_abs = np.abs(coh_post - coh_pre)
        coh_prior = coh_prob[0].detach().cpu().numpy()
        true_mask = labels[0].detach().cpu().numpy()
        pred_mask = preds[0].detach().cpu().numpy()
        fig, axes = plt.subplots(1, 7, figsize=(28, 4))
        axes[0].imshow(image); axes[0].set_title("RGB"); axes[0].axis("off")
        axes[1].imshow(coh_pre, cmap="gray", vmin=0, vmax=1); axes[1].set_title("COH_pre"); axes[1].axis("off")
        axes[2].imshow(coh_post, cmap="gray", vmin=0, vmax=1); axes[2].set_title("COH_post"); axes[2].axis("off")
        axes[3].imshow(coh_diff_abs, cmap="magma", vmin=0, vmax=1); axes[3].set_title("|COH_post-COH_pre|"); axes[3].axis("off")
        axes[4].imshow(coh_prior, cmap="viridis", vmin=0, vmax=1); axes[4].set_title("Coherence prior"); axes[4].axis("off")
        axes[5].imshow(true_mask, cmap="gray", vmin=0, vmax=1); axes[5].set_title("Ground Truth"); axes[5].axis("off")
        axes[6].imshow(pred_mask, cmap="gray", vmin=0, vmax=1); axes[6].set_title("Prediction"); axes[6].axis("off")
        plt.tight_layout(); plt.show()

class SaveMetricsCallback(TrainerCallback):
    def __init__(self, output_file):
        self.output_file = output_file
        self.metrics = []
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None:
            row = {"step": state.global_step}
            row.update(metrics)
            self.metrics.append(row)
    def on_train_end(self, args, state, control, **kwargs):
        if len(self.metrics) == 0:
            print("No evaluation metrics to save.")
            return
        with open(self.output_file, "w", newline="") as csvfile:
            fieldnames = list(self.metrics[0].keys())
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.metrics)
        print("Metrics saved:", self.output_file)


In [ ]:
# =========================================================
# CELL 11 — Training arguments
# =========================================================

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_total_limit=3,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=1000,
    eval_steps=1000,
    logging_steps=10,
    eval_accumulation_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="Recall",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED
)

In [ ]:
# =========================================================
# CELL 12 — Trainer
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
    callbacks=[DisplayCallback(), SaveMetricsCallback(metrics_file_path)]
)
print("Trainer ready.")


In [ ]:
# =========================================================
# CELL 13 — Train and save best model
# =========================================================

trainer.train()
os.makedirs(best_model_dir, exist_ok=True)
trainer.save_model(best_model_dir)
image_processor.save_pretrained(best_model_dir)

print("\nTraining finished.")
print("Best model saved in:")
print(best_model_dir)


In [ ]:
# =========================================================
# CELL 14 — Final validation evaluation
# =========================================================

val_results = trainer.evaluate(valid_ds)

print("\nValidation results:")
for key, value in val_results.items():
    print(f"{key}: {value}")

pd.DataFrame([val_results]).to_csv(
    os.path.join(output_dir, "validation_results.csv"),
    index=False
)

print("\nIndependent test evaluation is performed in the inference notebook.")